
# <center> **THE VLOGGER BIG FIVE COMPETITION** 

The YouTube Personality Dataset is a comprehensive compilation of behavioral 
attributes, speech transcriptions, and personality assessment scores for a group 
of **404 YouTube vloggers**. These creators share their thoughts while facing 
the camera, discussing a range of subjects such as personal experiences, 
political matters, movies, literature, etc.

Within this notebook, we attempt to predict the **personality traits** of these YouTubers.

# **1.0 IMPORT DATA**

To do so we start by importing the following data and packages:

> - Transcripts
> - Personality scores
> - Gender
> - Audio and visual information

> - tidyverse
> - tidytext

## **1.1 CREATE PATH**

In the next step we create a path to the **.txt** and **.csv** files.


In [ ]:
# Loading packages
library(tidyverse) 
library(tidytext)

# Listing files
list.files(path = "../input")

# Directory structure 
master_dir = file.path(list.files('../input', full.names=TRUE), 'youtube-personality')
directory_content = list.files(master_dir, full.names = TRUE)
print(directory_content)

# Path to the transcripts directory with transcript .txt files
path_to_transcripts = directory_content[2] 

# .csv filenames
AudioVisual_file    = directory_content[3]
Gender_file         = directory_content[4]
Personality_file    = directory_content[5]


## **1.2 TRANSCRIPTS**

Next, we want to import the transcripts of the vlogs and encode their ID, which is stored as **vlogID**. Then, we create a **tibble** to represent our data. The data frama includes the **vlogID**, the text of the transcripts, and the filename.

Let's have a look at it!



In [ ]:
# Importing the transcripts
transcript_files = list.files(path_to_transcripts, full.names = TRUE) 
print(head(transcript_files))

# Encoding the ID
vlogId = basename(transcript_files)
vlogId = str_replace(vlogId, pattern = ".txt$", replacement = "")
head(vlogId)

# Including features extracted from the transcript texts
transcripts_df = tibble(
    
    # vlogId connects each transcripts to a vlogger
    vlogId=vlogId,
    
    # Read the transcript text from all file and store as a string
    TEXT = map_chr(transcript_files, ~ paste(readLines(.x), collapse = "\\n")), 
    
    # `filename` keeps track of the specific video transcript
    filename = transcript_files
)

# Inspecting the first two examples
transcripts_df %>% 
    head(2)

## **1.2 PERSONALITY SCORES**

Now we import the personality scores based on the **Big Five**. 

> The **Big Five**, as described by Goldberg (1993), includes five personality traits that capture variations in people's thoughts, feelings, and actions. These traits are **neuroticism, extraversion, openness, agreeableness,** and **conscientiousness**.

We import them with the following commands and save them as **pers_df**. Now we can also look at them.

In [ ]:
# Import the Personality scores
pers_df = read_delim(Personality_file, delim=" ")

# Checking the data frame
head(pers_df)

## **1.3 GENDER**

> According to Costa et al. (2001), there is a pattern in which women score higher on the **Big Five** for **agreeableness** and **neuroticism**. Therefore, gender could be of interest. 

We extract it using these commands and add a column name.

In [ ]:
# Storing gender info
gender_df = read.delim(Gender_file, head=FALSE, sep=" ", skip = 2)

# Add column names
names(gender_df) = c('vlogId', 'gender')

# preview data
head(gender_df)

## **1.4 MERGE DATAFRAMES**

Now we combine the **gender** and **pers** dataframe to have one dataset.

We also dummy code the variable **gender**. The value 0 represents male now and 1 represents female.

In [ ]:
# merging gender and pers with left_join
vlogger_df01 = left_join(gender_df, pers_df, by='vlogId')

# switching characters for numeric values for gender variable
vlogger_df01$gender[vlogger_df01$gender == "Female"] <- 1
vlogger_df01$gender[vlogger_df01$gender == "Male"] <- 0

# preview data
head(vlogger_df01)

>  Some rows have **missing values**. This is because they are part of what will later be known as the **test set**. But before we split the dataset. Let's look at the transcripts again.

## **1.5. AUDIO VISUAL INFORMATION**

> Our manner of speaking often serves as a window into our inner world, unveiling facets of our character that might otherwise remain concealed. Privious research therefore suggests that speech may be a way of assessing personality (Polzehl, 2015).

Luckily we have access to some of the **audio and nonverbal cues** of the videos. So let's have a look at them.

In [ ]:
# Importing the audio and visual information
AudioVisual_df <- read.delim(AudioVisual_file, sep = " ", head=TRUE)

# preview data
head(AudioVisual_df)

We see that there a quite a few audio and nonverbal cues. Do we really need all of them?

## **1.6 MERGE DATAFRAMES**

Now let's combine only those **audio and visual cues** with our dataframe that are the average of something and start with **"mean"**.

In [ ]:
# select all columns containing the mean 
selected_columns <- grep("^mean", names(AudioVisual_df), value = TRUE)

# Join the selected columns to vlogger_df01
vlogger_df1 <- left_join(vlogger_df01, select(AudioVisual_df, vlogId, all_of(selected_columns)), by = 'vlogId')

# Print the first few rows of the resulting data frame
head(vlogger_df1)

# **2. FEATURES FROM TRANSCRIPTS**

Let us extract some features from the text that we can use to predict the **personality scores** of the **test set**.

> Our approach follows the steps explained in: **[Introducing Text Analytics](https://www.kaggle.com/code/datasniffer/introducing-text-analytics-big-5-from-text)**

## **2.1 TOKENS FROM TEXT**

We begin our feature extraction by tokenizing the vlogs into single **words** so we can analyze their meaning later on. 

To represent our data we have chosen a wordcloud this time. We see the 150 most popular words in our dataset. A lot of "and", right?

In [ ]:
# Tokenizing the transcripts into words
transcript_features_df = 
    transcripts_df %>%
    unnest_tokens(token, TEXT, token = 'words') 

# Creating a wordcloud to see the 100 most used words
transcript_features_df %>%
    mutate(token = tolower(token)) %>%
count(token, sort= TRUE) %>%
with(., wordcloud::wordcloud(token, n, max.words = 100))

## **2.2 MERGE DATAFRAMES**

We are now going to merge **transcript_feature_df** with **vlogger_df_1**, which is the data frame we created when we merged gender and personality scores earlier on.



In [ ]:
# Merging transcript_features_df with vlogger_df1
vlogger_df = left_join(vlogger_df1, transcript_features_df, by='vlogId') %>%
   select(-filename)

# preview of data
head(vlogger_df)

## **2.3 NRC EMOTION LEXICON**
In this section we will add the lexicons from the **textdata** package. 

We will use the **NRC Emotion Lexicon**. The NRC Emotion Lexicon is a list of English words and their associations with eight basic emotions (**anger, fear, anticipation, trust, surprise, sadness, joy,** and **disgust**) and two sentiments (**negative** and **positive**).

We will compute scores for each vlogger that indicate how often they used words associated with the eight basic emotions.

In [ ]:
# adding lexicons from 'textdata' package
get_lexicon = function(lexicon_name = names(textdata:::download_functions)) {
    lexicon_name = match.arg(lexicon_name)
    textdata:::download_functions[[lexicon_name]]('.')
    rds_filename = paste0(lexicon_name,'.rds')
    textdata:::process_functions[[lexicon_name]]('.',rds_filename)
    readr::read_rds(rds_filename)
}

# computing sentiment scores as features
nrc <- get_lexicon('nrc')
transcript_sentiment <- left_join(vlogger_df, nrc,  by = c(token = 'word'), relationship='many-to-many')%>% 
count(`vlogId`, sentiment) %>%
pivot_wider(id_cols = 'vlogId', names_from=sentiment, values_from=n, values_fill = 0) 

## **2.2 REMOVING STOPWORDS**

Remeber how we saw a lot of "and" before on the word cloud? We will now remove stopwords from our data.

> **Stopwords** are common words (e.g., "the," "and," "in") that carry little meaningful information. We decided to remove stopwords, as we believe they do not help in detecting the personality (Das & Das, 2016).

We created a **wordcloud** again.

In [ ]:
# getting stopwords
stopwords <- get_stopwords() 

# removing stopwords
transcript_features_df <- transcript_features_df %>%
    anti_join(stopwords, by = c(token = "word"))

#creating a wordcloud to see the 100 most used words without stopwords
transcript_features_df %>%
count(token, sort= TRUE) %>%
with(., wordcloud::wordcloud(token, n, max.words = 100))

## **2.3 MERGING DATA**

Now we can merge our data frame and look at it. Notice how we see a count of the words associated with the eight basic emotions (**anger, fear, anticipation, trust, surprise, sadness, joy,** and **disgust**) and two sentiments (**negative** and **positive**) from the NRC Emotion Lexicon list.

In [ ]:
# merging our dataframe 
data01 <- inner_join(vlogger_df1,transcript_sentiment, by = "vlogId")

# preview data
head(data01)

## **2.4 SENTENCE LENGTH**

> As previously mentioned, the way we speak says something about us. Therefore, we decided to add sentence length as a feature, as it may be an indicator of communication style. 

We count the **sentences** per vloger and add them to our data frame by merging it again.

In [ ]:
# make a column that counts the length of a sentence by summing the words per vlog.
transcript_features_df_2 <- 
    transcripts_df %>%
    unnest_tokens(sentences, TEXT, token = 'sentences') %>%
  group_by(vlogId) %>%
  summarize(sentences = n()) %>%
  as_tibble()

# Merge the two data frames
data02 <- inner_join(data01, transcript_features_df_2, by = "vlogId")

# preview data
head(data02)

## **2.4. WORD LENGTH**

> Not only the length of a sentence may be an important aspect of our communication style but also the length of words.

Therfore we added a feature that counts the **long words** (i.e. all words that have more than 11 characters) for each vlogger.

In [ ]:
# make a dataframe that counts the letters in each word:
word_length <- transcript_features_df %>%
    select(vlogId, token) %>%
    mutate(token = nchar(token))

# objectivly calculate what short and long words should be by calculating std
summary(word_length$token)

# creating a variable that contains the length of the words
word_length <- word_length %>%
    mutate(long_word = token >= 11)

# creating a variable that contains long words
long_word_df <- word_length %>%
  group_by(vlogId) %>%
  summarize(long_words = sum(long_word))

# now adding the feature to the data 
data <- data02 %>%
  left_join(long_word_df, by = "vlogId")

# preview data
head(data)

# **3. PREDICTIVE MODEL**

In this section we will try out some models and explain what results we got and why that may be the case. 

We also calculate the the **Root Means Square Error** for each model based on the training data.

>  $\displaystyle{RMSE =\sqrt{{1 \over 5n} \sum_{k \in \{cEXT, \ldots, cOPN\}} \sum_{i=1}^n (y_{ik} - \hat y_{ik})^2}}$


## **3.1 ADDITIVE MODEL**

Our first try is an additive model with all features as predictors. When we look at the output we see that for some personality types we get a significant F-statistic, while for others we don't. Another interesting thing we observe is that some features appear to be significant predictors for some personality types but not for others. Overall, this model has a **RSME** of 0.742 on the training data.

In [ ]:
# building an additive model
mod_01 <- lm(cbind(Extr, Agr, Cons, Emot, Open) ~ gender + mean.pitch + mean.conf.pitch + mean.spec.entropy + mean.val.apeak + mean.loc.apeak + mean.num.apeak + mean.num.apeak + mean.energy + mean.d.energy + anger + anticipation + disgust + fear + joy + negative + positive + sadness + surprise + trust + sentences + long_words, data = data)

# print model summary
summary(mod_01)

# RMSE for the training dataset
(RMSE_01 <- sqrt(mean(mod_01$residuals^2)))

## **3.2. ADDITIVE MODEL REVISION**

Our first additive model did not perform too bad, but let's see if we can improve it. Therefore, we will try to build an additve model that only includes those features that are significant predictors for at least one personality type. That means we remove the following features: **mean pitch, disgust, negative, sadness,** and **trust**. 

When we look at the output we see that nowe we do get significant F-statistic for most personality types. However the **RMSE** for this model is 0.754 on the training data.

In [ ]:
# building a second additive model
mod_02 <- lm(cbind(Extr, Agr, Cons, Emot, Open) ~ gender + mean.conf.pitch + mean.spec.entropy + mean.val.apeak + mean.loc.apeak + mean.num.apeak + mean.num.apeak + mean.energy + mean.d.energy + anger + anticipation + fear + joy + positive + surprise + sentences + long_words, data = data)

# print model summary
summary(mod_02)

# RMSE for the training dataset
(RMSE_02 <- sqrt(mean(mod_02$residuals^2)))

## **3.3. INTERACTIVE MODEL**

Since reducing the number of parameters in our additive model did not lead to improved predictions, we have chosen to take a different approach. We are adding interactions between features that share common characteristics, such as pairing word meanings with a positive connotation and word types with a negative connotation. The intention behind this is to introduce more predictors into our model, which, in theory, might enhance its accuracy.

However, our findings with this approach have been unexpected. While the **RMSE** for the training data decreased to 0.651, upon submitting our predictions, we noticed that the **RMSE** for the test data increased to 0.845. How can this be explained?

> It's crucial to grasp that the addition of interactions increases the complexity of our model. This complexity enables the model to fit the training data exceptionally well, capturing even noise and minor variations within the training set. However, this advantage comes at a cost – the model may become excessively tailored to the training data, which can lead to a poorer fit when applied to the test data. This phenomenon is commonly known as **overfitting** (Dietterich, 1995).

In [ ]:
# building an interactive model
mod_03 <- lm(cbind(Extr, Agr, Cons, Emot, Open) ~ gender + mean.pitch * mean.conf.pitch + mean.spec.entropy + mean.val.apeak * mean.loc.apeak * mean.num.apeak * mean.num.apeak + mean.energy * mean.d.energy + anger * disgust * fear * negative * sadness + anticipation * joy * positive * surprise * trust + sentences * long_words, data = data)

# print model summary
summary(mod_03)

# RMSE for the training dataset
(RMSE_03 <- sqrt(mean(mod_03$residuals^2)))

## **3.4 MODEL BASED ON CORRELATION MATRIX**

As we found that enabling interactions between the features in our model did not lead to an improvement in prediction accuracy, we will now examine the correlation matrix to identify any variables with high correlation.

> When variables exhibit strong correlations with each other, we refer to this as **multicollinearity**. Multicollinearity is a data issue that can significantly impact the reliability of the model parameter estimates (Alin, 2010).
>
> Also, highly correlated variables form a risk of overfitting since their presence is unnecessary and can have adverse impact on the model (Cook & Ranstam, 2016)

As a result, we aim to eliminate highly correlated variables from our dataset and then build an additive model with the remaining features. This model yield a **RMSE** of 0.762 on the training set.

In [ ]:
# Subsetting the features that could be correlated
corr_data <- data[, 8:28]

# Compute the correlation matrix 
correlation_matrix <- cor(corr_data)

# Find highly correlated variables
highly_correlated_vars <- caret::findCorrelation(correlation_matrix, cutoff = 0.8)

# Print the highly correlated variables
print(highly_correlated_vars)

# Remove highly correlated variables from the correlation subset
corr_data <- corr_data[, -highly_correlated_vars]

# Remove the original columns 1 to 7 from the 'data' dataframe
data_cor <- data[, 1:7]

# Combine the modified 'corr_data' with the 'data' dataframe
data_cor <- cbind(data, corr_data)

# preview data
head(data_cor)

# Display the column names of the updated 'data' dataframe
colnames(data_cor)

In [ ]:
# building an additive model with remaining features
mod_04 <- lm(cbind(Extr, Agr, Cons, Emot, Open) ~ gender + mean.pitch + mean.conf.pitch + mean.spec.entropy + mean.val.apeak + mean.loc.apeak + mean.energy + mean.d.energy + disgust + fear + joy + surprise + sentences + long_words, data = data_cor)

# print model summary
summary(mod_04)

# RMSE for the training dataset
(RMSE_04 <- sqrt(mean(mod_04$residuals^2)))

## **3.5 HYBRID STEPWISE REGRESSION**

So far our best prediction was achieved with the additive model including all features. Instead of keeping our "try and error" approach to find a better model we can try to use hybrid stepwise regression to evaluate the best model based on AIC.

> The Akaike information criterion (Akaike, 1973) is a popular method for comparing the adequacy of multiple, possibly nonnested models. IC is calculated from:
>
> * the number of independent variables used to build the model.
> * the maximum likelihood estimate of the model.
>
> The best-fit model according to AIC is the one that explains the greatest amount of variation using the fewest possible features.

> There are three options when doing stepwise regression which are 'forward', 'backward' and 'both'. Both is often regarded as a compromise between agressive selection of 'forward' and the computational intensitiy of 'backward'.

We start by defining our dependent variables, which are the Big Five. Then, we create a list to store our stepwise models. In the next step we perform our hybrid stepwise linear regression. Our best model is selected based on the smallest AIC. We can see this model stored as model 5. However, the models **RMSE** is 0.790 on the training data. 

In [ ]:
# List of response variables
response_vars <- c("Extr", "Agr", "Cons", "Emot", "Open")

# Create an empty list to store the stepwise models
stepwise_models <- list()

# Iterate over each response variable and fit a stepwise model
for (response_var in response_vars) {
  # Create the formula for the current response variable
  formula_str <- paste(response_var, "~ gender + mean.pitch + mean.conf.pitch + mean.spec.entropy + 
  mean.val.apeak + mean.loc.apeak + mean.num.apeak + mean.num.apeak + mean.energy + mean.d.energy + 
  anger + anticipation + disgust + fear + joy + negative + positive + sadness + surprise + trust + 
  sentences + long_words")
  lm_formula <- as.formula(formula_str)
  
  # Fit the linear model
  lm_model <- lm(lm_formula, data = data)
  
  # Perform stepwise regression
  stepwise_model <- step(lm_model, direction = "both")
  
  # Store the stepwise model in the list
  stepwise_models[[response_var]] <- stepwise_model
}

# Access and view the summaries of the stepwise models
for (response_var in response_vars) {
  cat("Summary for response variable:", response_var, "\n")
  summary(stepwise_models[[response_var]])
  cat("\n")
}

# best model based on smallest AIC
mod_05 <- lm(cbind(Extr, Agr, Cons, Emot, Open) ~ mean.loc.apeak + mean.num.apeak + anticipation + joy + 
    positive + sadness + trust, data = data)

# RMSE for the training dataset
(RMSE_05 <- sqrt(mean(mod_05$residuals^2)))

## **3.6 AUDIO AND VISUAL FEATURES ALONE**

As we could not get a better prediction based on the lowest **AIC**, let's try something else. Maybe the meaning of the words used in the Youtube videos isn't really relevant for predicting the personality types. 

Therefore, let us build a model that only includes the **audio and visual features**. The calculated **RMSE** for this model is 0.805 on the training data.

In [ ]:
# building a model only including audio and visual features
mod_6 <- lm(cbind(Extr, Agr, Cons, Emot, Open) ~ mean.pitch + mean.conf.pitch + mean.spec.entropy + mean.val.apeak + mean.loc.apeak + mean.num.apeak + mean.num.apeak + mean.energy + mean.d.energy, data = data)

# print model summary
summary(mod_6)

# RMSE for the training dataset
(RMSE_06 <- sqrt(mean(mod_6$residuals^2)))

## **3.7 TEXT FEATURES ALONE**

Using only audio and visual features yielded a relatively high **RMSE**. Let's investigate if we can achieve better predictions by exclusively utilizing the features extracted from the transcripts.

It appears we might be onto something promising. Our predictions have shown improvement! Our new **RMSE** is 0.776, but how can we account for this?

> Well, when we contemplate audio and visual features such as pitch, energy, etc., we must acknowledge that the way we speak is highly idiosyncratic. Perhaps even excessively so? It's plausible that the audio and visual features could **lack of generalizability**, hence it might be prudent to exclude them.
>
>  Further, research has shown that people's lexicon holds a lot of valuable information about their (Big Five) personality (Argamon, Dhawle, Koppel & Pennebaker, 2005). Thus, using only the vlogger's words to predict their personality seems like a valid option.


In [ ]:
# building a model only including text features
mod_07 <- lm(cbind(Extr, Agr, Cons, Emot, Open) ~ anger + anticipation + disgust + fear + joy + negative + positive + sadness + surprise + trust + sentences + long_words, data = data)

# print model summary
summary(mod_07)

# RMSE for the training dataset
(RMSE_07 <- sqrt(mean(mod_07$residuals^2)))

## **3.8 VISUALIZING RMSE**

Now, we can create a plot to visualize the **RMSE** values from the training data. In this plot, it becomes evident that **model 3** exhibits the lowest **RMSE** for the test data. However, as previously mentioned, this model does not perform well when applied to the actual test data. Consequently, we will proceed with **model 7**.

In [ ]:
# combine RMSEs into one vector:
all_rmse <- c(RMSE_01, RMSE_02, RMSE_03, RMSE_04, RMSE_05, RMSE_06, RMSE_07)

# Names of the RMSE values 
names <- c("MOD_01", "MOD_02", "MOD_03", "MOD_04", "MOD_05", "MOD_06", "MOD_07")

# Create a barplot with names on the X-axis and values on the Y-axis
barplot(all_rmse, names.arg = names, main = "RMSE Values", xlab = "RMSE Names", ylab = "RMSE Values", col = "grey")



# **4. TEST DATA**

> Remember the **test set**? 

We can now create a subset from our data that contains those observations for which the personality traits were not observed. This data will be our **test set**. With the function **nrow** we can see that this dataset consists of **80 observations**. 

In [ ]:
# Creating a subset for test data
test_data = data %>% 
    filter(is.na(Extr))

# counting rows
nrow(test_data)

## **4.2 USING PREDICTIONS**

Now we are using our **prediction model 07** to predict the personality values in our **test set**.


In [ ]:
# Prediction for test data
pred_mod_all = predict(mod_07, new = test_data)

# Compute output data frame
test_data_pred = test_data %>% 
    mutate(
        Extr = pred_mod_all[,'Extr'], 
        Agr  = pred_mod_all[,'Agr' ],
        Cons = pred_mod_all[,'Cons'],
        Emot = pred_mod_all[,'Emot'],
        Open = pred_mod_all[,'Open']
    ) %>%
    select(vlogId, Extr:Open)

# preview of data
head(test_data_pred)

## **4.3 CHANGE FILE**

To **export our predictions** to a **.csv** file we need to transform it to **long format**! To do this we can use **pivot_longer**. Then we want to save them so that we have only two coloums. Therefore we add the **personality descriptions** to the **vlogID**.

In [ ]:
# Converting our data to long format
test_data_pred_long <- test_data_pred %>% 
    pivot_longer(c(Extr, Agr, Cons, Emot, Open), names_to='pers_axis')

# Obtain the right format for Kaggle
test_data_pred_final <- test_data_pred_long %>%
    unite(Id, vlogId, pers_axis) %>%
    rename(Expected = value)

# Check if we succeeded
head(test_data_pred_final)

In [ ]:
# Write to csv
write_csv(test_data_pred_final, file = "predictions_07.csv")

# Check if the file was written successfully.
dir()

# **5. ACKNOWLEDGMENTS** 

The transcripts used for our prediction were manually transcribed by a professional company. The transcription corresponds to the full video duration. In total, around 28h of video were annotated. 
The data was anonymized by substituting any name entities referenced that could be use to identify the video or the vlogger with a ‘XXXX'. The transcriptions are provided in raw text and contain a total of ~10K unique words and ~240K word tokens.

**Reference for transcripts:**

> Biel, J.-I., Tsiminaki, V., Dines, J., & Gatica-Perez, D. (2013). Hi YouTube!: Personality Impressions and Verbal Content in Social Video. In Proceedings of the 15th ACM International Conference on Multimodal Interaction (pp. 119-126). ACM.

**Further Reference:**

>Akaike, H. (1974). A new look at the statistical model identification.IEEE Transactions on Automatic Control,19, 716–723.
>
>Alin, A. (2010). Multicollinearity. Wiley interdisciplinary reviews: computational statistics, 2(3), 370-374.
>
> Argamon, S., Dhawle, S., Koppel, M., & Pennebaker, J. W. (2005). Lexical predictors of personality type. In Proceedings of the 2005 joint annual meeting of the interface and the classification society of North America (pp. 1-16).
>
> Das, K. G., & Das, D. (2017). Developing lexicon and classifier for personality identification in texts. In Proceedings of the 14th International Conference on Natural Language Processing (pp. 362-372).
>
> Dietterich, T. (1995). Overfitting and undercomputing in machine learning. ACM computing surveys (CSUR), 27(3), 326-327.
> 
> Costa, P. T. Jr., Terracciano, A., and McCrae, R. R. (2001). Gender differences in personality traits across cultures: robust and surprising findings. J. Pers. Soc. Psychol. 81, 322–331.
> 
> Goldberg, L.R. (1993) The Structure of Phenotypic Personality Traits. American Psychologist, 48, 26-34. http://dx.doi.org/10.1037/0003-066X.48.1.26
>
>Pennebaker, J. W. (2011). Your use of pronouns reveals your personality. Harvard Business Review, 89(12), 32-33.
>
> Polzehl, T. (2015). Personality in speech. In T-labs series in telecommunication services. https://doi.org/10.1007/978-3-319-09516-5
>
> Cook, J., & Ranstam, J. (2016). Overfitting. British Journal of Surgery, 103(13), 1814. https://doi.org/10.1002/bjs.10244




# **6. DIVISION OF LABOR**

>*  **Loading Data:** Laura
>*  **Features:** Roman and Laura
>*  **Predicitve Model:** Bram and Laura
>*  **Test data:** Laura
>*  **Acknowledgments:** Laura
>*  **Text, Design & Visualizations:** Bram and Laura


In [ ]:

# 1. Define project paths
project_name <- "Kaggle_Project"
project_dir <- file.path("/kaggle/working", project_name)

dir.create(project_dir, recursive = TRUE, showWarnings = FALSE)

# 2. Copy source datasets
data_dir <- file.path(project_dir, "data")
dir.create(data_dir, showWarnings = FALSE)

if (dir.exists("/kaggle/input")) {
  system2(
    "cp",
    c("-R", "/kaggle/input/.", shQuote(data_dir))
  )
}

# 3. Copy generated outputs
output_dir <- file.path(project_dir, "outputs")
dir.create(output_dir, showWarnings = FALSE)

files <- list.files(
  "/kaggle/working",
  full.names = TRUE,
  all.files = TRUE,
  no.. = TRUE
)

files <- files[
  !basename(files) %in% c(
    project_name,
    paste0(project_name, ".zip")
  )
]

file.copy(
  files,
  output_dir,
  recursive = TRUE,
  overwrite = TRUE
)

# 4. Create ZIP archive
zip_path <- file.path(
  "/kaggle/working",
  paste0(project_name, ".zip")
)

if (file.exists(zip_path)) file.remove(zip_path)

system2(
  "zip",
  c("-rq", shQuote(zip_path), shQuote(project_name)),
  stdout = TRUE
)

# 5. Verify ZIP creation
if (file.exists(zip_path)) {
  cat("ZIP created successfully:\n", zip_path)
  cat("\nDownload it from the Kaggle Output files.")
} else {
  stop("ZIP creation failed.")
}
